# 03 — Exploratory Data Analysis

## tl;dr

This notebook examines overall demand, stores, product families, promotions, holidays, transactions, and oil prices. It prioritizes interpretable patterns and operational relevance. All relationships are descriptive; promotion, holiday, transaction, and oil comparisons are not causal estimates.

## Context & Methods

High-level analysis uses the full sales panel. Detailed store-family views use the justified subset selected in Notebook 02. Daily averages are preferred when month length or partial periods could distort totals.

### Key assumptions

- Sales aggregate mixed unit types across product families, so portfolio-level totals are an index of activity rather than a common physical unit.
- Promotion comparisons are exposed to selection bias because promotions are not randomly assigned.
- Sales per transaction is a descriptive basket proxy, not revenue or margin per customer.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
# Resolve the repository root dynamically so the notebook works from either
# the project root or the notebooks directory without a local absolute path.
candidate = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (candidate, *candidate.parents)
    if (path / "data").is_dir() and (path / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DATA_DIR = PROJECT_ROOT / "outputs" / "processed_data"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
for directory in (PROCESSED_DATA_DIR, OUTPUT_DATA_DIR, FIGURE_DIR, TABLE_DIR):
    # Creating output folders is safe; raw source files are never modified.
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from src.data_loader import load_all_available
from src.preprocessing import build_analysis_panel
from src.eda import (
    classify_demand_patterns,
    analyze_payday_effects,
    analyze_earthquake_impact,
    compute_cross_family_correlations,
    analyze_promotion_elasticity,
)
from src.visualization import (
    PROJECT_COLORS,
    save_figure,
    set_project_style,
    plot_demand_classification,
    plot_payday_comparison,
    plot_earthquake_impact,
    plot_correlation_heatmap,
)

set_project_style()
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
datasets = load_all_available(RAW_DATA_DIR)
panel = build_analysis_panel(datasets)
scope = json.loads((OUTPUT_DATA_DIR / "analysis_scope.json").read_text())
selected_stores = scope["selected_stores"]
selected_families = scope["selected_families"]

## Results

### 1. Overall demand trend and rolling average

**Why this step is needed:** The network trend establishes the broad direction of observed sales, while a rolling average reduces day-to-day noise so sustained changes are easier to see.

In [ ]:
daily_total = panel.groupby("date", observed=True)["sales"].sum().sort_index()
rolling_28 = daily_total.rolling(28, min_periods=14).mean()
fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(daily_total.index, daily_total, color="#9bb7d3", linewidth=.7, label="Daily sales")
ax.plot(rolling_28.index, rolling_28, color=PROJECT_COLORS[0], linewidth=2, label="28-day moving average")
ax.set(title="Total observed sales over time", xlabel="Date", ylabel="Sales (mixed units)")
ax.legend()
save_figure(fig, FIGURE_DIR, "eda_01_total_sales_trend.png")
plt.show()

### 2. Year, month, and weekday patterns

**Why this step is needed:** Calendar patterns reveal recurring workload differences that may affect staffing, delivery schedules, receiving capacity, and replenishment cadence.

In [ ]:
calendar_daily = daily_total.rename("sales").reset_index()
calendar_daily["year"] = calendar_daily["date"].dt.year
calendar_daily["month"] = calendar_daily["date"].dt.month
calendar_daily["day_of_week"] = calendar_daily["date"].dt.dayofweek

yearly = calendar_daily.groupby("year").agg(total_sales=("sales", "sum"), mean_daily_sales=("sales", "mean"), days=("date", "nunique"))
monthly = calendar_daily.groupby("month")["sales"].mean()
weekday = calendar_daily.groupby("day_of_week")["sales"].mean().reindex(range(7))
weekday.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
display(yearly)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
monthly.plot(kind="bar", ax=axes[0], color=PROJECT_COLORS[0])
axes[0].set(title="Average daily sales by calendar month", xlabel="Month", ylabel="Average daily sales")
weekday.plot(kind="bar", ax=axes[1], color=PROJECT_COLORS[1])
axes[1].set(title="Average daily sales by weekday", xlabel="Weekday", ylabel="Average daily sales")
fig.tight_layout()
save_figure(fig, FIGURE_DIR, "eda_02_calendar_patterns.png")
plt.show()

### 3. Store ranking, type, cluster, and geography

**Why this step is needed:** Store totals alone do not explain operating context. Adding format, cluster, geography, traffic coverage, and variability supports more defensible comparisons.

In [ ]:
store_daily = panel.groupby(["store_nbr", "date"], observed=True).agg(
    sales=("sales", "sum"), transactions=("transactions", "max")
).reset_index()
store_summary = store_daily.groupby("store_nbr").agg(
    total_sales=("sales", "sum"),
    mean_daily_sales=("sales", "mean"),
    std_daily_sales=("sales", "std"),
    transaction_coverage_pct=("transactions", lambda values: 100 * values.notna().mean()),
    total_transactions=("transactions", "sum"),
)
store_summary["cv"] = store_summary["std_daily_sales"] / store_summary["mean_daily_sales"]
store_summary["sales_per_transaction_proxy"] = store_summary["total_sales"] / store_summary["total_transactions"]
store_summary = store_summary.join(datasets["stores.csv"].set_index("store_nbr"))
display(store_summary.nlargest(10, "total_sales"))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
store_summary.nlargest(15, "total_sales").sort_values("total_sales")["total_sales"].plot(
    kind="barh", ax=axes[0], color=PROJECT_COLORS[0]
)
axes[0].set(title="Top 15 stores by total observed sales", xlabel="Sales (mixed units)", ylabel="Store")
store_summary.groupby("type", observed=True)["mean_daily_sales"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color=PROJECT_COLORS[1]
)
axes[1].set(title="Average daily store sales by store type", xlabel="Average daily sales", ylabel="Store type")
fig.tight_layout()
save_figure(fig, FIGURE_DIR, "eda_03_store_performance.png")
plt.show()

geo_summary = store_summary.groupby("state", observed=True).agg(
    stores=("total_sales", "size"), total_sales=("total_sales", "sum"), mean_store_daily_sales=("mean_daily_sales", "mean")
).sort_values("total_sales", ascending=False)
cluster_summary = store_summary.groupby("cluster", observed=True).agg(
    stores=("total_sales", "size"), mean_store_daily_sales=("mean_daily_sales", "mean"), mean_cv=("cv", "mean")
).sort_values("mean_store_daily_sales", ascending=False)
display(geo_summary.head(10))
display(cluster_summary)

### 4. Product-family importance, variability, and growth

**Why this step is needed:** Volume indicates commercial importance, whereas variability indicates planning difficulty. Both are needed to avoid treating high-sales and high-uncertainty categories as the same problem.

In [ ]:
family_daily = panel.groupby(["family", "date"], observed=True)["sales"].sum().reset_index()
family_summary = family_daily.groupby("family", observed=True)["sales"].agg(
    total_sales="sum", mean_daily_sales="mean", std_daily_sales="std"
)
family_summary["cv"] = family_summary["std_daily_sales"] / family_summary["mean_daily_sales"]
family_summary["sales_share_pct"] = 100 * family_summary["total_sales"] / family_summary["total_sales"].sum()

first_year = panel["year"].min()
last_full_year = panel.loc[panel["year"].lt(panel["year"].max()), "year"].max()
family_year = panel.groupby(["family", "year"], observed=True)["sales"].sum().unstack(fill_value=0)
family_summary["growth_first_to_last_full_year_pct"] = 100 * (
    family_year[last_full_year] / family_year[first_year].replace(0, np.nan) - 1
)
display(family_summary.sort_values("total_sales", ascending=False).head(15))

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
family_summary.nlargest(15, "total_sales").sort_values("total_sales")["total_sales"].plot(
    kind="barh", ax=axes[0], color=PROJECT_COLORS[0]
)
axes[0].set(title="Top 15 product families by total sales", xlabel="Sales (mixed units)", ylabel="Family")
axes[1].scatter(family_summary["mean_daily_sales"], family_summary["cv"], color=PROJECT_COLORS[4], alpha=.8)
for family in family_summary.nlargest(5, "total_sales").index:
    axes[1].annotate(family, (family_summary.loc[family, "mean_daily_sales"], family_summary.loc[family, "cv"]), fontsize=8)
axes[1].set(title="Family demand level versus variability", xlabel="Mean daily sales", ylabel="Coefficient of variation")
fig.tight_layout()
save_figure(fig, FIGURE_DIR, "eda_04_family_importance_variability.png")
plt.show()

### 5. Detailed store-family comparison for the justified subset

**Why this step is needed:** A store-family view reveals differences hidden by network totals and creates a manageable visual for an MBA presentation.

In [ ]:
scope_panel = panel.loc[
    panel["store_nbr"].isin(selected_stores) & panel["family"].isin(selected_families)
]
store_family = scope_panel.groupby(["store_nbr", "family"], observed=True)["sales"].sum().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(12, 6))
image = ax.imshow(store_family, aspect="auto", cmap="Blues")
ax.set_xticks(range(len(store_family.columns)), labels=store_family.columns, rotation=45, ha="right")
ax.set_yticks(range(len(store_family.index)), labels=store_family.index)
ax.set(title="Sales concentration across selected stores and families", xlabel="Product family", ylabel="Store")
fig.colorbar(image, ax=ax, label="Total sales (mixed units)")
fig.tight_layout()
save_figure(fig, FIGURE_DIR, "eda_05_store_family_heatmap.png")
plt.show()

### 6. Promotion association and heterogeneity

**Why this step is needed:** Comparing promoted and non-promoted observations identifies where operational coordination may matter, while remaining explicitly descriptive because promotions were not randomly assigned.

In [ ]:
promotion_comparison = panel.assign(promoted=panel["onpromotion"].gt(0)).groupby(
    ["family", "promoted"], observed=True
)["sales"].mean().unstack()
promotion_comparison.columns = ["mean_sales_not_promoted", "mean_sales_promoted"]
promotion_comparison["promoted_to_nonpromoted_ratio"] = (
    promotion_comparison["mean_sales_promoted"] / promotion_comparison["mean_sales_not_promoted"].replace(0, np.nan)
)
promotion_comparison["promotion_coverage_pct"] = panel.groupby("family", observed=True)["onpromotion"].apply(
    lambda values: 100 * values.gt(0).mean()
)
display(promotion_comparison.sort_values("promoted_to_nonpromoted_ratio", ascending=False).head(15))

plot_promo = promotion_comparison.loc[family_summary.nlargest(12, "total_sales").index].sort_values("promoted_to_nonpromoted_ratio")
fig, ax = plt.subplots(figsize=(11, 7))
plot_promo["promoted_to_nonpromoted_ratio"].plot(kind="barh", ax=ax, color=PROJECT_COLORS[3])
ax.axvline(1, color="black", linestyle="--", linewidth=1)
ax.set(title="Observed promoted/non-promoted mean-sales ratio — top families", xlabel="Ratio (association, not causal lift)", ylabel="Family")
save_figure(fig, FIGURE_DIR, "eda_06_promotion_association.png")
plt.show()

store_promo = panel.assign(promoted=panel["onpromotion"].gt(0)).groupby(
    ["store_nbr", "promoted"], observed=True
)["sales"].mean().unstack()
store_promo["ratio"] = store_promo[True] / store_promo[False].replace(0, np.nan)
display(store_promo["ratio"].sort_values(ascending=False).head(10).rename("promoted_to_nonpromoted_ratio"))

### 7. Holidays and major events

**Why this step is needed:** Holiday effects can differ by national, regional, and local scope. Scope-aware comparisons help identify potential peak periods without misapplying local events to every store.

In [ ]:
store_date = panel.groupby(["store_nbr", "date"], observed=True).agg(
    sales=("sales", "sum"),
    is_holiday=("is_holiday", "max"),
    is_national_holiday=("is_national_holiday", "max"),
    is_regional_holiday=("is_regional_holiday", "max"),
    is_local_holiday=("is_local_holiday", "max"),
).reset_index()
holiday_comparison = store_date.groupby("is_holiday")["sales"].agg(["count", "mean", "median", "std"])
display(holiday_comparison)

scope_means = {}
for column in ("is_national_holiday", "is_regional_holiday", "is_local_holiday"):
    scope_means[column] = store_date.groupby(column)["sales"].mean().to_dict()
display(pd.DataFrame(scope_means).T.rename(columns={False: "normal_or_other", True: "flagged_scope"}))

daily_event = datasets["holidays_events.csv"].loc[
    ~datasets["holidays_events.csv"]["transferred"]
].groupby("date")["description"].agg(lambda values: " | ".join(sorted(set(values))))
daily_sales = daily_total.to_frame().join(daily_event).dropna()
display(daily_sales.nlargest(12, "sales"))

### 8. Transactions, traffic, and the sales-per-transaction proxy

**Why this step is needed:** Transactions provide a store-traffic proxy that helps separate changes in customer visits from changes in the mixed-unit sales measure.

In [ ]:
traffic = panel.groupby(["date", "store_nbr"], observed=True).agg(
    sales=("sales", "sum"), transactions=("transactions", "max")
).dropna().reset_index()
traffic["sales_per_transaction_proxy"] = traffic["sales"] / traffic["transactions"].replace(0, np.nan)
traffic_correlation = traffic[["sales", "transactions"]].corr().iloc[0, 1]
print(f"Store-day Pearson correlation between sales and transactions: {traffic_correlation:.3f}")

sample = traffic.sample(min(12000, len(traffic)), random_state=RANDOM_SEED)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(sample["transactions"], sample["sales"], s=8, alpha=.18, color=PROJECT_COLORS[0])
axes[0].set(title="Store-day transactions and observed sales", xlabel="Transactions", ylabel="Sales (mixed units)")
store_summary.sort_values("sales_per_transaction_proxy")["sales_per_transaction_proxy"].plot(
    kind="bar", ax=axes[1], color=PROJECT_COLORS[1]
)
axes[1].set(title="Sales-per-transaction proxy by store", xlabel="Store", ylabel="Mixed sales units per transaction")
axes[1].tick_params(axis="x", labelsize=7)
fig.tight_layout()
save_figure(fig, FIGURE_DIR, "eda_07_transactions.png")
plt.show()

### 9. Oil prices as exploratory context

**Why this step is needed:** Oil is a possible macroeconomic context variable, but a visual and correlation are used only to screen for patterns—not to claim that oil prices cause demand changes.

In [ ]:
oil_daily = panel.groupby("date", observed=True).agg(
    sales=("sales", "sum"), dcoilwtico=("dcoilwtico", "max")
).reset_index()
oil_monthly = oil_daily.set_index("date").resample("MS").mean()
oil_corr = oil_monthly[["sales", "dcoilwtico"]].corr().iloc[0, 1]
print(f"Monthly contemporaneous correlation: {oil_corr:.3f} (descriptive only)")

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(oil_monthly.index, oil_monthly["sales"], color=PROJECT_COLORS[0], label="Average daily sales")
ax1.set(xlabel="Month", ylabel="Average daily sales")
ax2 = ax1.twinx()
ax2.plot(oil_monthly.index, oil_monthly["dcoilwtico"], color=PROJECT_COLORS[4], label="WTI oil price", alpha=.8)
ax2.set_ylabel("WTI price")
ax1.set_title("Sales and oil prices over time — exploratory context")
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [line.get_label() for line in lines], loc="upper left")
save_figure(fig, FIGURE_DIR, "eda_08_oil_context.png")
plt.show()

### 10. Demand pattern classification (Syntetos-Boylan / Croston)

**Why this step is needed:** Retail demand varies not only by volume but by regularity. Categorizing demand streams into Smooth, Intermittent, Erratic, and Lumpy quadrants guides the choice of inventory policy and forecasting models.

In [ ]:
demand_classification = classify_demand_patterns(panel, group_cols=["family"])
display(demand_classification.sort_values(["demand_category", "adi"]))
fig = plot_demand_classification(demand_classification, output_dir=FIGURE_DIR)
plt.show()

### 11. Semi-monthly payday dynamics (Quincenas: 15th & month-end)

**Why this step is needed:** In Ecuador, formal wage disbursements occur on the 15th and at month-end. Identifying liquidity-driven surges helps stores schedule stock replenishment and checkout staffing.

In [ ]:
payday_results = analyze_payday_effects(panel)
display(payday_results["summary"])
display(payday_results["by_family"].head(10))
fig = plot_payday_comparison(payday_results["by_family"], top_n=12, output_dir=FIGURE_DIR)
plt.show()

### 12. Structural shock analysis: April 16, 2016 earthquake

**Why this step is needed:** A major Mw 7.8 earthquake hit Ecuador on April 16, 2016. Analyzing pre- vs post-earthquake shifts across epicenter provinces (Manabí, Esmeraldas) vs non-impacted areas reveals disaster response patterns and category demand shifts.

In [ ]:
quake_results = analyze_earthquake_impact(panel, window_days=30)
display(quake_results["regional_impact"].head(10))
display(quake_results["family_shift"].head(10))
fig = plot_earthquake_impact(quake_results, output_dir=FIGURE_DIR)
plt.show()

### 13. Cross-category demand correlation & co-movement

**Why this step is needed:** Evaluating cross-category correlation identifies complementary product categories that experience simultaneous demand spikes versus independent categories.

In [ ]:
corr_matrix = compute_cross_family_correlations(panel)
top_volume_families = list(family_summary.nlargest(12, "total_sales").index)
fig = plot_correlation_heatmap(corr_matrix, top_families=top_volume_families, output_dir=FIGURE_DIR)
plt.show()

### 14. Save compact evidence tables

**Why this step is needed:** Persisting reviewed summaries lets operations, forecasting, and managerial notebooks reuse the same evidence rather than recomputing or manually copying values.

In [ ]:
store_summary.reset_index().to_csv(TABLE_DIR / "store_summary.csv", index=False)
family_summary.reset_index().to_csv(TABLE_DIR / "family_summary.csv", index=False)
promotion_comparison.reset_index().to_csv(TABLE_DIR / "promotion_summary.csv", index=False)
yearly.reset_index().to_csv(TABLE_DIR / "yearly_summary.csv", index=False)
holiday_comparison.reset_index().to_csv(TABLE_DIR / "holiday_summary.csv", index=False)
demand_classification.to_csv(TABLE_DIR / "demand_classification_summary.csv", index=False)
payday_results["by_family"].reset_index().to_csv(TABLE_DIR / "payday_summary.csv", index=False)
quake_results["regional_impact"].reset_index().to_csv(TABLE_DIR / "earthquake_regional_impact.csv", index=False)
corr_matrix.reset_index().to_csv(TABLE_DIR / "cross_family_correlations.csv", index=False)
print("Saved nine compact evidence tables to reports/tables/.")

## Takeaways

In [ ]:
top_store = store_summary["total_sales"].idxmax()
top_family = family_summary["total_sales"].idxmax()
peak_weekday = weekday.idxmax()
peak_month = int(monthly.idxmax())
smooth_families = demand_classification.loc[demand_classification["demand_category"] == "Smooth", "family"].count()
lumpy_families = demand_classification.loc[demand_classification["demand_category"] == "Lumpy", "family"].count()
display(Markdown(f'''
- **Store {top_store}** has the highest total observed sales; store scale and variability differ materially across the network.
- **{top_family}** contributes the most family-level sales, while family CV shows that importance and predictability are separate operational dimensions.
- Average daily demand is highest on **{peak_weekday}** and in calendar month **{peak_month}**, which motivates differentiated staffing and replenishment review.
- Promoted rows often have higher mean sales, but the ratio varies by family and store and must be treated as association rather than incremental causal lift.
- Holiday effects vary by scope; date-only joins would obscure local and regional relevance.
- Transactions and sales move together (store-day correlation **{traffic_correlation:.3f}**), but the mixed-unit sales-per-transaction measure is only a basket proxy.
- The monthly sales/oil correlation is **{oil_corr:.3f}** and is exploratory; shared trends and omitted factors prevent causal interpretation.
- **Demand Classification:** {smooth_families} categories exhibit smooth demand (ideal for standard forecasting), while {lumpy_families} categories exhibit lumpy demand (requiring specialized safety stock or Croston intermittent forecasting).
- **Quincena Payday Effect:** Significant demand surge occurs on the 15th and month-end dates, confirming a recurring bi-weekly liquidity pattern across stores.
- **External Shock:** The April 2016 earthquake caused pronounced category shifts (surge in relief essentials like food/water vs. discretionary declines) with sharp regional divergence.
'''))